# 레슨 03 — HTML 테이블과 리스트 데이터 정리

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/03/%5B%ED%95%99%EC%83%9D%EC%9A%A9%5D%20%EB%A0%88%EC%8A%A8%2003%20%E2%80%94%20HTML%20%ED%85%8C%EC%9D%B4%EB%B8%94%EA%B3%BC%20%EB%A6%AC%EC%8A%A4%ED%8A%B8%20%EB%8D%B0%EC%9D%B4%ED%84%B0%20%EC%A0%95%EB%A6%AC.ipynb)

이 노트북은 읽기와 따라하기용 강의 노트북이다. 학생은 셀을 위에서 아래로 실행하며 입력 데이터가 어떤 구조로 바뀌는지 확인한다. HTML 테이블과 리스트 데이터 정리는 실제 업무 자동화에서 자주 등장하는 반복 패턴을 합성 fixture로 안전하게 연습한다.

## 학습 목표

1. HTML table의 thead/tbody 구조를 읽는다.
2. tr과 td를 딕셔너리로 변환한다.
3. 카드형 UI와 리스트형 UI를 별도로 파싱한다.
4. 여러 구조에서 나온 값을 하나의 요약으로 합친다.
5. 정리한 데이터를 CSV로 저장한다.

---

## 1. 테이블 구조 읽기

테이블은 th 헤더와 td 값을 같은 순서로 묶는 것이 핵심이다.


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/03/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))


print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

html_text = load_text('class_dashboard.html')
soup = BeautifulSoup(html_text, 'html.parser')
headers = [th.text.strip() for th in soup.select('#class-table thead th')]
rows = soup.select('#class-table tbody tr')
print(headers, len(rows))



---

## 2. 행을 record로 변환

각 tr을 dict로 바꾸면 필터링과 정렬이 쉬워진다.


In [ ]:
records = []
for tr in rows:
    cells = [td.text.strip() for td in tr.select('td')]
    records.append(dict(zip(headers, cells)))
print(records[0])



---

## 3. 카드와 리스트 구조

article.feedback-card와 li.todo-item은 table과 다른 반복 단위다.


In [ ]:
feedback_soup = BeautifulSoup(load_text('feedback_cards.html'), 'html.parser')
urgent = [card.select_one('.title').text.strip() for card in feedback_soup.select('article.feedback-card') if card['data-priority'] == 'high']
print(urgent)



---

## 데이터 출처와 안전 규칙

이 레슨의 파일은 모두 수업용 합성 데이터다. 실제 사이트의 개인정보, 로그인 정보, 유료 콘텐츠를 포함하지 않는다. 실제 사이트로 확장할 때는 robots.txt, 이용 약관, 요청 간격, 수집 목적을 먼저 확인한다. 수업 중에는 fixture를 반복 실행하며 구조를 익히고, 외부 사이트를 빠르게 반복 요청하지 않는다.

---

## 강의 보강 노트

이 절은 수업 중 교사가 질문으로 풀어낼 수 있는 운영형 설명이다. 학생이 셀을 실행한 뒤 결과만 맞히지 않고 자동화 절차를 말로 설명하도록 돕는다.

### 1. 표와 카드의 차이

HTML table은 행과 열의 의미가 비교적 명확하지만 card/list UI는 사람이 보기 좋게 흩어진 정보를 다시 구조화해야 한다. 학생에게 먼저 반복 단위가 `tr`인지 `.card`인지 찾게 하면 파싱 코드가 자연스럽게 정리된다.

### 2. 헤더 매핑

표 데이터를 읽을 때 열 순서에만 의존하면 컬럼이 추가될 때 코드가 깨진다. 헤더 텍스트를 key로 만들고 각 행의 cell을 매핑하는 방식은 실제 대시보드 추출에서 더 안정적인 패턴이다.

### 3. 리스트 정규화

카드, 표, ul/li에서 뽑은 데이터는 최종적으로 같은 딕셔너리 구조로 맞춰야 비교와 저장이 가능하다. 학생 답안에서는 필드 이름이 서로 다르게 흩어져 있지 않은지 확인한다.

### 4. 빈 값 처리

웹 화면에는 빈 칸, 숨김 태그, 안내 문구가 섞인다. 빈 문자열을 그대로 저장할지, `None`으로 바꿀지, 오류로 볼지 기준을 정해 두어야 데이터 품질을 설명할 수 있다.

### 5. 선택자 검증

selector가 맞는지는 첫 값 출력보다 `len()`으로 개수를 확인하는 편이 좋다. 개수가 예상보다 많거나 적으면 반복 단위를 잘못 잡았을 가능성이 높다는 점을 수업 중 반복해서 묻는다.

### 6. 정렬과 요약

수집한 표를 그대로 보여주는 데서 끝내지 말고 상태별 개수, 점수 평균, 우선순위 높은 항목 같은 작은 요약을 만들게 한다. 자동화 결과가 운영자에게 의미를 가지려면 요약 기준이 필요하다.

### 7. 저장 전 리뷰

CSV로 저장하기 전에 첫 3행과 마지막 3행을 확인하는 습관을 둔다. 학생이 만든 데이터가 사람이 읽을 수 있는 이름과 순서를 갖고 있는지 보는 것도 채점 포인트다.

### 체크포인트 1: 저장 경로 요약한다

재실행했을 때 같은 결과를 얻기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 저장 경로을/를 요약한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 2: 실행 순서 검토한다

다음 셀에서 재사용하기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 실행 순서을/를 검토한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 3: 운영 메모 정리한다

실제 사이트 확장 전에 위험을 낮추기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 운영 메모을/를 정리한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 4: 출력 형태 설명한다

저장 파일의 신뢰도를 높이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 출력 형태을/를 설명한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 5: 반복 단위 저장한다

코랩과 로컬 실행 차이를 줄이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 반복 단위을/를 저장한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 6: 상태 값 되돌아본다

수업 중 피드백 시간을 줄이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 상태 값을/를 되돌아본다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 7: 저장 경로 표준화한다

학생이 막힌 지점을 찾기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 저장 경로을/를 표준화한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 8: 실행 순서 검증한다

운영자가 결과를 이해할 수 있게, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 실행 순서을/를 검증한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 9: 운영 메모 확인한다

재실행했을 때 같은 결과를 얻기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 운영 메모을/를 확인한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 10: 출력 형태 분리한다

다음 셀에서 재사용하기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 출력 형태을/를 분리한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 11: 반복 단위 기록한다

실제 사이트 확장 전에 위험을 낮추기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 반복 단위을/를 기록한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 12: 상태 값 비교한다

저장 파일의 신뢰도를 높이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 상태 값을/를 비교한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 13: 저장 경로 요약한다

코랩과 로컬 실행 차이를 줄이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 저장 경로을/를 요약한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 14: 실행 순서 검토한다

수업 중 피드백 시간을 줄이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 실행 순서을/를 검토한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 15: 운영 메모 정리한다

학생이 막힌 지점을 찾기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 운영 메모을/를 정리한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 16: 출력 형태 설명한다

운영자가 결과를 이해할 수 있게, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 출력 형태을/를 설명한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 17: 반복 단위 저장한다

재실행했을 때 같은 결과를 얻기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 반복 단위을/를 저장한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 18: 상태 값 되돌아본다

다음 셀에서 재사용하기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 상태 값을/를 되돌아본다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 19: 저장 경로 표준화한다

실제 사이트 확장 전에 위험을 낮추기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 저장 경로을/를 표준화한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 20: 실행 순서 검증한다

저장 파일의 신뢰도를 높이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 실행 순서을/를 검증한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 21: 운영 메모 확인한다

코랩과 로컬 실행 차이를 줄이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 운영 메모을/를 확인한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 22: 출력 형태 분리한다

수업 중 피드백 시간을 줄이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 출력 형태을/를 분리한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 23: 반복 단위 기록한다

학생이 막힌 지점을 찾기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 반복 단위을/를 기록한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 24: 상태 값 비교한다

운영자가 결과를 이해할 수 있게, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 상태 값을/를 비교한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 25: 저장 경로 요약한다

재실행했을 때 같은 결과를 얻기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 저장 경로을/를 요약한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 26: 실행 순서 검토한다

다음 셀에서 재사용하기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 실행 순서을/를 검토한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 27: 운영 메모 정리한다

실제 사이트 확장 전에 위험을 낮추기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 운영 메모을/를 정리한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 28: 출력 형태 설명한다

저장 파일의 신뢰도를 높이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 출력 형태을/를 설명한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 29: 반복 단위 저장한다

코랩과 로컬 실행 차이를 줄이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 반복 단위을/를 저장한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 30: 상태 값 되돌아본다

수업 중 피드백 시간을 줄이기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 상태 값을/를 되돌아본다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 31: 저장 경로 표준화한다

학생이 막힌 지점을 찾기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 저장 경로을/를 표준화한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 32: 실행 순서 검증한다

운영자가 결과를 이해할 수 있게, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 실행 순서을/를 검증한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 33: 운영 메모 확인한다

재실행했을 때 같은 결과를 얻기 위해, 레슨 03 HTML 테이블과 리스트 데이터 정리에서는 운영 메모을/를 확인한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.
